In [3]:
import pandas as pd
import numpy as np

mycsv = 'backtest_results.csv'
df = pd.read_csv(mycsv)
print(f"{mycsv} successfully ")
print(df.head())

backtest_results.csv successfully 
          Strategy        Entry Time         Exit Time  Type  Lots  \
0  LONDON_BREAKOUT  09/09/2025 06:25  09/09/2025 06:35   BUY  0.02   
1    RSI_REVERSION  09/09/2025 13:30  09/09/2025 13:55  SELL  0.01   
2   VWAP_REVERSION  09/09/2025 13:55  09/09/2025 14:00  SELL  0.01   
3        SMC_CHOCH  09/09/2025 22:25  09/09/2025 22:50  SELL  0.01   
4    RSI_REVERSION  10/09/2025 00:45  10/09/2025 01:50   BUY  0.01   

  Entry Price   Exit Price    RR     PnL  Balance  
0    3638.609     3643.901  2.00   10.58   510.58  
1    3663.623  3667.769351  2.91   -4.15   506.44  
2    3666.057  3670.121631  4.14   -4.06   502.37  
3    3624.063   3634.63484  2.00  -10.57   491.80  
4    3623.297  3630.321286  1.56    7.02   498.83  


In [2]:
df_clean = df.iloc[:-2].copy()
df_clean.tail()

,Strategy,Entry Time,Exit Time,Type,Lots,Entry Price,Exit Price,RR,PnL,Balance
587,RSI_REVERSION,29/05/2026 14:55,29/05/2026 15:00,SELL,0.02,4563.997,4571.810188,2.84,-15.63,1348.31
588,SMC_CHOCH,01/06/2026 01:00,01/06/2026 06:35,SELL,0.01,4534.135,4502.234582,2.00,31.9,1380.21
589,VWAP_REVERSION,01/06/2026 13:10,01/06/2026 13:20,BUY,0.02,4483.901,4476.450231,3.73,-14.9,1365.31
590,RSI_REVERSION,01/06/2026 13:20,01/06/2026 13:50,BUY,0.02,4465.903,4458.294228,4.21,-15.22,1350.09
591,SMC_CHOCH,01/06/2026 14:00,01/06/2026 17:40,SELL,0.01,4454.261,4486.402452,2.00,-32.14,1317.95


In [5]:
#convert entry date and exit date to date time object

df_clean["Entry Time"] = pd.to_datetime(df_clean["Entry Time"], format="%d/%m/%Y %H:%M")
df_clean["Exit Time"] = pd.to_datetime(df_clean["Exit Time"], format="%d/%m/%Y %H:%M")

In [9]:
df_clean['entry_hour'] = df_clean["Entry Time"].dt.hour

#mapping to market session
def get_Session(hour):
    if 0<= hour < 7:
        return "Asian"
    elif 7<= hour < 13:
        return "London"
    elif 13 <= hour < 20:
        return "New York"
    else:
        return "Late NY / Off-hours"


df_clean['session'] = df_clean["entry_hour"].apply(get_Session)


# Calculate duration in minutes

df_clean["duration_mins"] = (df_clean["Exit Time"] - df_clean["Entry Time"]).dt.total_seconds() / 60


In [12]:
# str to numeric conversion
numeric_cols = ["Entry Price", "Exit Price", "Lots", "RR", "PnL", "Balance"]
for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col])
print(df_clean.dtypes)

Strategy                    str
Entry Time       datetime64[us]
Exit Time        datetime64[us]
Type                        str
Lots                    float64
Entry Price             float64
Exit Price              float64
RR                      float64
PnL                     float64
Balance                 float64
entry_hour                int32
session                     str
duration_mins           float64
dtype: object


In [13]:
session_summary = df_clean.groupby("session").agg(
    total_trades=("PnL", "count"),
    win_rate = ("PnL",lambda x: f"{(x > 0 ).mean() * 100:.1f}%"),
    total_pnl = ("PnL","sum"),
    avg_pnl = ("PnL","mean")
).reset_index()

print(session_summary)

               session  total_trades win_rate  total_pnl   avg_pnl
0                Asian           219    37.4%     184.92  0.844384
1  Late NY / Off-hours           116    37.1%     286.93  2.473534
2               London           106    41.5%     886.42  8.362453
3             New York           151    23.2%    -540.28 -3.578013


In [14]:
#Now we need to isolate the root cause: Is New York bad for every strategy, or is there a specific strategy failing during US market hours?

#We can answer this using a Pivot Table in Pandas.
# 1. Total PnL Pivot Table
session_strategy_pnl = df_clean.pivot_table(
    index="Strategy",
    columns="session",
    values="PnL",
    aggfunc="sum",
    fill_value=0
).round(2)

print("Net PnL by Strategy across Sessions:")
print(session_strategy_pnl)

Net PnL by Strategy across Sessions:
session           Asian  Late NY / Off-hours  London  New York
Strategy                                                      
BB_BREAKOUT       -0.50                 0.00   -7.64    -34.11
LIQUIDITY_SWEEP  118.28                30.73   33.12      0.00
LONDON_BREAKOUT   10.58                 0.00    0.00      0.00
RSI_REVERSION    109.42               -33.87   64.65   -146.33
SMC_CHOCH       -318.71               178.39  387.15    -61.71
SMC_FVG          250.08                42.47  179.73    -66.14
SMC_OB             0.00               -12.31   -6.22     22.36
VWAP_REVERSION    15.77                81.52  235.63   -254.35


In [16]:
#Create a second pivot table replacing aggfunc="sum" with aggfunc="count" to see how many trades each strategy takes in each session.

session_strategy_pnl = df_clean.pivot_table(
    index="Strategy",
    columns="session",
    values="PnL",
    aggfunc="count",
    fill_value=0
).round(2)

print("count trade by Strategy across Sessions:")
print(session_strategy_pnl)

count trade by Strategy across Sessions:
session          Asian  Late NY / Off-hours  London  New York
Strategy                                                     
BB_BREAKOUT          3                    0       4         3
LIQUIDITY_SWEEP      3                    3       1         0
LONDON_BREAKOUT      1                    0       0         0
RSI_REVERSION       48                   25      11        23
SMC_CHOCH           90                   61      40        53
SMC_FVG             51                   12      16        20
SMC_OB               0                    4       1         1
VWAP_REVERSION      23                   11      33        51


In [17]:
#What would the account balance and total profit look like if we simply disabled trading during the New York session?
df_no_ny = df_clean[df_clean['session'] != "New York" ].copy()

# 2. Compare Total PnL
original_pnl = df_clean['PnL'].sum()
filtered_pnl = df_no_ny['PnL'].sum()

#compare win rates
original_win_rate = (df_clean['PnL'] > 0).mean() * 100
filtered_win_rate = (df_no_ny['PnL'] > 0).mean() * 100

print(f"Original Trades: {len(df_clean)} | Filtered Trades: {len(df_no_ny)}")
print(f"Original PnL: ${original_pnl:.2f}  ==>  Filtered PnL (No NY): ${filtered_pnl:.2f}")
print(f"Original Win Rate: {original_win_rate:.1f}%  ==>  Filtered Win Rate: {filtered_win_rate:.1f}%")

Original Trades: 592 | Filtered Trades: 441
Original PnL: $817.99  ==>  Filtered PnL (No NY): $1358.27
Original Win Rate: 34.5%  ==>  Filtered Win Rate: 38.3%


In [18]:
# Apply both rule filters
df_rule2 = df_clean[
    (df_clean["session"] != "New York") & 
    ~((df_clean["Strategy"] == "SMC_CHOCH") & (df_clean["session"] == "Asian"))
].copy()

# Recalculate Performance Metrics
pnl_rule2 = df_rule2["PnL"].sum()
win_rate_rule2 = (df_rule2["PnL"] > 0).mean() * 100
pf_rule2 = df_rule2[df_rule2["PnL"] > 0]["PnL"].sum() / abs(df_rule2[df_rule2["PnL"] < 0]["PnL"].sum())

print(f"Original Trades: {len(df_clean)}  ==>  Optimized Trades: {len(df_rule2)}")
print(f"Original PnL: ${original_pnl:.2f}     ==>  Optimized PnL: ${pnl_rule2:.2f}")
print(f"Original Win Rate: {original_win_rate:.1f}%  ==>  Optimized Win Rate: {win_rate_rule2:.1f}%")
print(f"Optimized Profit Factor: {pf_rule2:.2f}")

Original Trades: 592  ==>  Optimized Trades: 351
Original PnL: $817.99     ==>  Optimized PnL: $1676.98
Original Win Rate: 34.5%  ==>  Optimized Win Rate: 39.3%
Optimized Profit Factor: 1.61


In [19]:
import plotly.graph_objects as go

# 1. Baseline Cumulative Equity
df_clean["baseline_equity"] = 500.0 + df_clean["PnL"].cumsum()

# 2. Optimized Cumulative Equity (Applying your session filters)
df_opt = df_clean[
    (df_clean["session"] != "New York") & 
    ~((df_clean["Strategy"] == "SMC_CHOCH") & (df_clean["session"] == "Asian"))
].copy()
df_opt["optimized_equity"] = 500.0 + df_opt["PnL"].cumsum()

# 3. Plot Comparison
fig_comp = go.Figure()

fig_comp.add_trace(go.Scatter(
    x=df_clean["Entry Time"],
    y=df_clean["baseline_equity"],
    mode="lines",
    name="Original Bot (All Sessions)",
    line=dict(color="#ef4444", width=1.8, dash="dot")
))

fig_comp.add_trace(go.Scatter(
    x=df_opt["Entry Time"],
    y=df_opt["optimized_equity"],
    mode="lines",
    name="Optimized Bot (Session Filtered)",
    line=dict(color="#10b981", width=2.5)
))

fig_comp.update_layout(
    title="<b>Equity Curve Comparison: Baseline vs. Optimized Rules</b>",
    xaxis_title="Date",
    yaxis_title="Account Balance ($)",
    template="plotly_dark",
    font=dict(family="monospace"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig_comp.show()